#### Side-by-side comparison of all **7 models** (5 ML + Bi-LSTM + Ensemble)

In [18]:
import numpy as np, pandas as pd, json, joblib, warnings
import matplotlib.pyplot as plt, seaborn as sns
import torch, torch.nn as nn
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             roc_curve, precision_recall_curve,
                             confusion_matrix, classification_report,
                             brier_score_loss)
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams.update({'figure.figsize': (10, 6), 'font.size': 11})
print("Libraries loaded.")

Libraries loaded.


In [19]:
DATA   = "../../data/processed/splitteddataset"
train_df = pd.read_csv(f"{DATA}/lepto_monthly_train.csv", parse_dates=["YearMonth"])
test_df  = pd.read_csv(f"{DATA}/lepto_monthly_test.csv",  parse_dates=["YearMonth"])

# Feature engineering (same as 03_Modeling)
le = joblib.load("../../models/label_encoder.pkl")
drr_df = pd.read_csv("../../models/district_risk_rate.csv")
drr_map = dict(zip(drr_df.iloc[:,0], drr_df.iloc[:,1]))

for df in [train_df, test_df]:
    df["District_enc"]       = le.transform(df["District"])
    df["District_risk_rate"] = df["District"].map(drr_map).astype(float)

DROP = ["District", "RiskLabel", "YearMonth"]
X_train = train_df.drop(columns=DROP)
y_train = train_df["RiskLabel"].values
X_test  = test_df.drop(columns=DROP)
y_test  = test_df["RiskLabel"].values

print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")

Train: (3900, 53)  |  Test: (1200, 53)


In [20]:
# Load saved ML models (fitted on full training data in 03_Modeling_And_Evaluation notebook)
model_names_ml = ["logreg", "rf", "xgb", "lgbm", "cat"]
display_names  = {"logreg": "Logistic Regression", "rf": "Random Forest",
                  "xgb": "XGBoost", "lgbm": "LightGBM", "cat": "CatBoost"}

ml_models = {}
for name in model_names_ml:
    try:
        ml_models[name] = joblib.load(f"../../models/{name}_model.pkl")
    except FileNotFoundError:
        pass

# the best_model is selected as RF from the above script
if not ml_models:
    ml_models["rf"] = joblib.load("../../models/best_model.pkl")
    print("Only RF model found. Will compute metrics for RF + Deep Learning models.")
else:
    print(f"Loaded {len(ml_models)} ML models: {list(ml_models.keys())}")

Loaded 5 ML models: ['logreg', 'rf', 'xgb', 'lgbm', 'cat']


#### LSTM Architecture (Same as 04_notebook)

In [21]:

class TemporalAttention(nn.Module):
    def __init__(self, hidden_dim, dropout=0.3):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)
        self.dropout = nn.Dropout(dropout) # Attention dropout added

    def forward(self, lstm_out):
        scores = self.attn(lstm_out)           
        weights = torch.softmax(scores, dim=1) 
        weights = self.dropout(weights) # Apply dropout to attention weights
        context = (weights * lstm_out).sum(dim=1)  
        return context, weights.squeeze(-1)

class LeptoLSTM_v2(nn.Module):
    def __init__(self, seq_input_dim, static_input_dim, hidden_dim=128, lstm_layers=2, dropout=0.3):
        super().__init__()
        self.conv = nn.Conv1d(in_channels=seq_input_dim, out_channels=seq_input_dim * 2, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.conv_dropout = nn.Dropout(dropout)

        self.lstm = nn.LSTM(
            input_size=seq_input_dim * 2,
            hidden_size=hidden_dim,
            num_layers=lstm_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout
        )
        bi_dim = hidden_dim * 2
        self.layer_norm = nn.LayerNorm(bi_dim) # LayerNorm added after LSTM

        self.attention = TemporalAttention(bi_dim, dropout=dropout)

        self.static_branch = nn.Sequential(
            nn.Linear(static_input_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 32),
            nn.ReLU()
        )

        self.head = nn.Sequential(
            nn.Linear(bi_dim + 32, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(dropout / 2),
            nn.Linear(64, 1)
        )

    def forward(self, x_seq, x_stat):
        x_seq_c = x_seq.transpose(1, 2)
        x_seq_c = self.conv(x_seq_c)
        x_seq_c = self.relu(x_seq_c)
        x_seq_c = self.conv_dropout(x_seq_c)
        
        x_seq_lstm = x_seq_c.transpose(1, 2)
        lstm_out, _ = self.lstm(x_seq_lstm)
        
        # Apply LayerNorm
        lstm_out = self.layer_norm(lstm_out)
        
        context, _ = self.attention(lstm_out)
        static_out  = self.static_branch(x_stat)
        combined    = torch.cat([context, static_out], dim=1)
        return self.head(combined)


In [22]:
with open('../../models/lstm_v2_meta.json', 'r') as f:
    meta = json.load(f)

lstm_model = LeptoLSTM_v2(meta['seq_input_dim'], meta['static_input_dim'])
lstm_model.load_state_dict(torch.load('../../models/lstm_v2_model.pth', map_location='cpu'))
lstm_model.eval()

sc_seq = joblib.load('../../models/lstm_v2_scaler_seq.pkl')
sc_stat = joblib.load('../../models/lstm_v2_scaler_stat.pkl')
print("LSTM weights and scalers loaded successfully!")


LSTM weights and scalers loaded successfully!


In [23]:
def build_sequences(df, meta):
    X_sq, X_st, y_out, idx_out = [], [], [], []
    for _, grp in df.groupby("District", observed=True):
        grp = grp.sort_values("YearMonth")
        idx = grp.index.tolist()
        sq = grp[meta["sequence_cols"]].values.astype(np.float32)
        st = grp[meta["static_cols"]].values.astype(np.float32)
        lb = grp["RiskLabel"].values
        for i in range(meta["seq_len"] - 1, len(grp)):
            X_sq.append(sq[i - meta["seq_len"] + 1 : i + 1])
            X_st.append(st[i])
            y_out.append(lb[i])
            idx_out.append(idx[i])
    return np.array(X_sq), np.array(X_st), np.array(y_out), idx_out

# Build for both train and test
Xsq_tr, Xst_tr, y_tr_lstm, idx_tr = build_sequences(train_df, meta)
Xsq_te, Xst_te, y_te_lstm, idx_te = build_sequences(test_df, meta)

print(f"LSTM Train Sequences: {Xsq_tr.shape}  |  Test Sequences: {Xsq_te.shape}")

LSTM Train Sequences: (3625, 12, 33)  |  Test Sequences: (925, 12, 33)


In [24]:
THRESHOLD = 0.45
results = {}

# ML Models
rf_model = ml_models.get("rf", ml_models.get(list(ml_models.keys())[0]))


for key, model in ml_models.items():
    # Reorder features
    if hasattr(model, "feature_names_in_"):
        X_tr_m = X_train[model.feature_names_in_]
        X_te_m = X_test[model.feature_names_in_]
    else:
        X_tr_m, X_te_m = X_train, X_test
    
    p_tr = model.predict_proba(X_tr_m)[:, 1]
    p_te = model.predict_proba(X_te_m)[:, 1]
    
    results[display_names.get(key, key)] = {
        "train_auc": roc_auc_score(y_train, p_tr),
        "test_auc":  roc_auc_score(y_test, p_te),
        "test_ap":   average_precision_score(y_test, p_te),
        "probs_te":  p_te,
        "probs_tr":  p_tr,
        "y_te":      y_test,
        "y_tr":      y_train,
    }

# LSTM v2
def lstm_predict(model, Xsq, Xst):
    Xsq_s = sc_seq.transform(Xsq.reshape(-1, Xsq.shape[2])).reshape(Xsq.shape)
    Xst_s = sc_stat.transform(Xst)
    with torch.no_grad():
        return torch.sigmoid(model(torch.tensor(Xsq_s), torch.tensor(Xst_s))).flatten().numpy()

p_lstm_tr = lstm_predict(lstm_model, Xsq_tr, Xst_tr)
p_lstm_te = lstm_predict(lstm_model, Xsq_te, Xst_te)

results["Bi-LSTM + Attention"] = {
    "train_auc": roc_auc_score(y_tr_lstm, p_lstm_tr),
    "test_auc":  roc_auc_score(y_te_lstm, p_lstm_te),
    "test_ap":   average_precision_score(y_te_lstm, p_lstm_te),
    "probs_te":  p_lstm_te,
    "probs_tr":  p_lstm_tr,
    "y_te":      y_te_lstm,
    "y_tr":      y_tr_lstm,
}

# Ensemble (RF + LSTM)
if hasattr(rf_model, "feature_names_in_"):
    X_rf_aligned_te = X_test.iloc[idx_te][rf_model.feature_names_in_]
    X_rf_aligned_tr = X_train.iloc[idx_tr][rf_model.feature_names_in_]
else:
    X_rf_aligned_te = X_test.iloc[idx_te]
    X_rf_aligned_tr = X_train.iloc[idx_tr]

p_rf_te_aligned = rf_model.predict_proba(X_rf_aligned_te)[:, 1]
p_rf_tr_aligned = rf_model.predict_proba(X_rf_aligned_tr)[:, 1]

w_lstm = 0.6
w_rf = 0.4

p_ens_te = w_lstm * p_lstm_te + w_rf * p_rf_te_aligned
p_ens_tr = w_lstm * p_lstm_tr + w_rf * p_rf_tr_aligned

results["Ensemble (RF + LSTM)"] = {
    "train_auc": roc_auc_score(y_tr_lstm, p_ens_tr),
    "test_auc":  roc_auc_score(y_te_lstm, p_ens_te),
    "test_ap":   average_precision_score(y_te_lstm, p_ens_te),
    "probs_te":  p_ens_te,
    "probs_tr":  p_ens_tr,
    "y_te":      y_te_lstm,
    "y_tr":      y_tr_lstm,
}

print("All predictions generated (Option 2 - Static Weights)")

All predictions generated (Option 2 - Static Weights)


#### Compare all the models

 - 5 ML Models
 - LSTM
 - Ensemble

In [25]:
rows = []
for name, r in results.items():
    p_te = r["probs_te"]
    y_te = r["y_te"]
    preds = (p_te >= THRESHOLD).astype(int)
    cm = confusion_matrix(y_te, preds)
    tn, fp, fn, tp = cm.ravel()
    
    rows.append({
        "Model": name,
        "Train AUC": r["train_auc"],
        "Test AUC":  r["test_auc"],
        "AUC Gap":   r["train_auc"] - r["test_auc"],
        "Test AP":   r["test_ap"],
        "Sensitivity": tp / (tp + fn),
        "Specificity": tn / (tn + fp),
        "F1 (HR)":     2*tp / (2*tp + fp + fn),
        "Brier Score":  brier_score_loss(y_te, p_te),
    })

comp_df = pd.DataFrame(rows).sort_values("Test AUC", ascending=False).reset_index(drop=True)
comp_df.index = comp_df.index + 1  # 1-indexed rank

print("=" * 80)
print("MASTER COMPARISON TABLE — All Models (threshold = 0.45)")
print("=" * 80)
print(comp_df.to_string(float_format="%.4f"))
print()

# Highlight the best
best = comp_df.iloc[0]
print(f"Best Model: {best['Model']}  (Test AUC = {best['Test AUC']:.4f})")

MASTER COMPARISON TABLE — All Models (threshold = 0.45)
                  Model  Train AUC  Test AUC  AUC Gap  Test AP  Sensitivity  Specificity  F1 (HR)  Brier Score
1  Ensemble (RF + LSTM)     0.9614    0.9121   0.0493   0.9171       0.8356       0.8212   0.8235       0.1398
2   Bi-LSTM + Attention     0.8954    0.9020  -0.0066   0.9080       0.8559       0.7609   0.8094       0.1534
3         Random Forest     0.9865    0.8990   0.0874   0.8980       0.8229       0.8077   0.8103       0.1320
4               XGBoost     0.8938    0.8948  -0.0010   0.8976       0.8038       0.8429   0.8144       0.1298
5              LightGBM     0.8939    0.8945  -0.0006   0.8971       0.8125       0.8317   0.8146       0.1299
6              CatBoost     0.8840    0.8919  -0.0080   0.8937       0.8125       0.8285   0.8132       0.1311
7   Logistic Regression     0.8739    0.8904  -0.0164   0.8916       0.7882       0.8317   0.8000       0.1319

Best Model: Ensemble (RF + LSTM)  (Test AUC = 0.9121)


#### OVERFITTING / UNDERFITTING ANALYSIS - ALL MODELS

In [26]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 2a. Train vs Test AUC Bar Chart
names = comp_df["Model"].tolist()
train_aucs = comp_df["Train AUC"].tolist()
test_aucs  = comp_df["Test AUC"].tolist()
gaps       = comp_df["AUC Gap"].tolist()

x = np.arange(len(names))
w = 0.35
bars1 = axes[0].barh(x - w/2, train_aucs, w, label="Train AUC", color="#60A5FA")
bars2 = axes[0].barh(x + w/2, test_aucs,  w, label="Test AUC",  color="#F87171")
axes[0].set_yticks(x)
axes[0].set_yticklabels(names, fontsize=9)
axes[0].set_xlim(0.85, 1.02)
axes[0].set_xlabel("AUC")
axes[0].set_title("Train vs Test AUC (Overfit Check)")
axes[0].legend()
axes[0].axvline(x=0.90, color="green", ls="--", alpha=0.5, label="0.90 target")

# 2b. AUC Gap
colors = ["#22C55E" if abs(g) < 0.05 else "#FBBF24" if abs(g) < 0.10 else "#EF4444" for g in gaps]
axes[1].barh(names, gaps, color=colors)
axes[1].set_xlabel("AUC Gap (Train - Test)")
axes[1].set_title("Generalisation Gap")
axes[1].axvline(x=0.05, color="orange", ls="--", alpha=0.7, label="Warning (0.05)")
axes[1].axvline(x=0.10, color="red",    ls="--", alpha=0.7, label="Overfit (0.10)")
axes[1].axvline(x=0.0,  color="gray",   ls="-",  alpha=0.3)
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig("../../reports/figures/fig_overfit_check.png", dpi=150, bbox_inches='tight')
plt.show()

print()
print("Overfitting Verdict:")
for _, row in comp_df.iterrows():
    gap = row["AUC Gap"]
    status = "Good fit" if abs(gap) < 0.05 else "Mild overfit" if abs(gap) < 0.10 else "Overfitting"
    print(f"  {row['Model']:30s}  Gap={gap:+.4f}  → {status}")


Overfitting Verdict:
  Ensemble (RF + LSTM)            Gap=+0.0493  → Good fit
  Bi-LSTM + Attention             Gap=-0.0066  → Good fit
  Random Forest                   Gap=+0.0874  → Mild overfit
  XGBoost                         Gap=-0.0010  → Good fit
  LightGBM                        Gap=-0.0006  → Good fit
  CatBoost                        Gap=-0.0080  → Good fit
  Logistic Regression             Gap=-0.0164  → Good fit


#### CALIBRATION CURVES - FINAL MODELS

In [27]:
fig, ax = plt.subplots(figsize=(8, 8))
ax.plot([0, 1], [0, 1], "k--", label="Perfectly calibrated")

# Only plot key models for clarity
key_models = ["Random Forest", "Bi-LSTM + Attention", "Ensemble (RF + LSTM)"]
colors_cal = {"Random Forest": "#378ADD", "Bi-LSTM + Attention": "#A78BFA", "Ensemble (RF + LSTM)": "#EF4444"}

for name in key_models:
    if name in results:
        r = results[name]
        fraction_pos, mean_pred = calibration_curve(r["y_te"], r["probs_te"], n_bins=10, strategy='uniform')
        brier = brier_score_loss(r["y_te"], r["probs_te"])
        ax.plot(mean_pred, fraction_pos, "o-", label=f"{name} (Brier={brier:.4f})", color=colors_cal.get(name, None))

ax.set_xlabel("Mean Predicted Probability")
ax.set_ylabel("Fraction of Positives")
ax.set_title("Calibration Plot (Reliability Diagram)")
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig("../../reports/figures/fig_calibration.png", dpi=150, bbox_inches='tight')
plt.show()

print("\nBrier Score (lower is better, 0 = perfect):")
for name in key_models:
    if name in results:
        bs = brier_score_loss(results[name]["y_te"], results[name]["probs_te"])
        print(f"  {name:30s}  Brier = {bs:.4f}")


Brier Score (lower is better, 0 = perfect):
  Random Forest                   Brier = 0.1320
  Bi-LSTM + Attention             Brier = 0.1534
  Ensemble (RF + LSTM)            Brier = 0.1398


#### ROC CURVES - ML BEST (RANDOM FOREST) + ENSEMBLE

In [30]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

selected_models = ["Ensemble (RF + LSTM)"]

palette = {
    #"Logistic Regression": "#6B7280",
    #"Random Forest":       "#DC2626",
    #"XGBoost":             "#3B82F6",
    #"LightGBM":            "#F59E0B",
    #"CatBoost":            "#EF4444",
    #"Bi-LSTM + Attention": "#8B5CF6",
    "Ensemble (RF + LSTM)":"#A78BFA",
}
# 4a. ROC
for name, r in results.items():
    if name not in selected_models:
        continue
    fpr, tpr, _ = roc_curve(r["y_te"], r["probs_te"])
    lw = 2
    axes[0].plot(fpr, tpr,
                 label=f"{name} ({r['test_auc']:.4f})",
                 color=palette.get(name),
                 lw=lw)
    axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR")
    axes[0].set_title("ROC Curves - Final Ensemble")
    axes[0].legend(fontsize=8, loc="lower right")

# 4b. Precision-Recall
for name, r in results.items():
    if name not in selected_models:
        continue
    prec, rec, _ = precision_recall_curve(r["y_te"], r["probs_te"])
    lw = 1
    axes[1].plot(rec, prec,
                 label=f"{name} (AP={r['test_ap']:.4f})",
                 color=palette.get(name),
                 lw=lw)
    axes[1].set_xlabel("Recall"); axes[1].set_ylabel("Precision")
    axes[1].set_title("Precision-Recall Curves - Final Ensemble")
    axes[1].legend(fontsize=8, loc="lower left")

    

plt.tight_layout()
plt.savefig("../../reports/figures/fig_all_models_roc_pr.png", dpi=150, bbox_inches='tight')
plt.show()

for name in ["Ensemble (RF + LSTM)"]:
    r = results[name]
    print(f"{name}: AUC={r['test_auc']:.4f}, AP={r['test_ap']:.4f}")

Ensemble (RF + LSTM): AUC=0.9121, AP=0.9171


#### ENSEMBLE CONFUSION MATRIX & METRICS

In [29]:
r_ens = results["Ensemble (RF + LSTM)"]
p_ens = r_ens["probs_te"]
y_ens = r_ens["y_te"]
preds_ens = (p_ens >= THRESHOLD).astype(int)
cm = confusion_matrix(y_ens, preds_ens)
tn, fp, fn, tp = cm.ravel()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Low Risk", "High Risk"],
            yticklabels=["Low Risk", "High Risk"], ax=axes[0], cbar=False)
axes[0].set_xlabel("Predicted"); axes[0].set_ylabel("Actual")
axes[0].set_title(f"Ensemble Confusion Matrix (t={THRESHOLD})")

# Metric bars
metrics = {"Sensitivity": tp/(tp+fn), "Specificity": tn/(tn+fp), "PPV": tp/(tp+fp), "NPV": tn/(tn+fn), "F1 HR": 2*tp/(2*tp+fp+fn)}
bars = axes[1].barh(list(metrics.keys()), list(metrics.values()), color=["#60A5FA","#34D399","#FBBF24","#F87171","#A78BFA"])
axes[1].set_xlim(0, 1)
axes[1].bar_label(bars, fmt="%.3f", padding=5)
axes[1].set_title("Ensemble Performance Metrics")

plt.tight_layout()
plt.savefig("../../reports/figures/fig_ensemble_cm.png", dpi=150, bbox_inches='tight')
plt.show()

print()
print(classification_report(y_ens, preds_ens, target_names=["Low Risk", "High Risk"]))


              precision    recall  f1-score   support

    Low Risk       0.84      0.82      0.83       481
   High Risk       0.81      0.84      0.82       444

    accuracy                           0.83       925
   macro avg       0.83      0.83      0.83       925
weighted avg       0.83      0.83      0.83       925




#### RESULTS SUMMARY

In [13]:

r_ens = results["Ensemble (RF + LSTM)"]
r_rf  = results.get("Random Forest", {})
r_lstm = results["Bi-LSTM + Attention"]

print("=" * 70)
print("THESIS RESULTS SUMMARY — Final Model Evaluation")
print("=" * 70)
print()
print(f"  {'Metric':<35s} {'RF':>10s} {'Bi-LSTM':>10s} {'Ensemble':>10s}")
print(f"  {'-'*65}")

def prt(label, rf_v, lstm_v, ens_v):
    rf_s   = f"{rf_v:.4f}" if rf_v else "N/A"
    lstm_s = f"{lstm_v:.4f}" if lstm_v else "N/A"
    ens_s  = f"{ens_v:.4f}" if ens_v else "N/A"
    print(f"  {label:<35s} {rf_s:>10s} {lstm_s:>10s} {ens_s:>10s}")

prt("Train AUC",   r_rf.get("train_auc"), r_lstm["train_auc"], r_ens["train_auc"])
prt("Test AUC",    r_rf.get("test_auc"),  r_lstm["test_auc"],  r_ens["test_auc"])
prt("AUC Gap",     r_rf.get("train_auc",0)-r_rf.get("test_auc",0), r_lstm["train_auc"]-r_lstm["test_auc"], r_ens["train_auc"]-r_ens["test_auc"])
prt("Test AP",     r_rf.get("test_ap"),   r_lstm["test_ap"],   r_ens["test_ap"])

# Per-model metrics
for name, r in [("RF", r_rf), ("Bi-LSTM", r_lstm), ("Ensemble", r_ens)]:
    if not r: continue
    p = r["probs_te"]
    y = r["y_te"]
    pred = (p >= THRESHOLD).astype(int)
    cm = confusion_matrix(y, pred)
    tn, fp, fn, tp = cm.ravel()

print()
print(f"  Overfitting Verdict (Ensemble):")
gap_ens = r_ens["train_auc"] - r_ens["test_auc"]
if abs(gap_ens) < 0.05:
    print(f"    AUC Gap = {gap_ens:+.4f} → NO overfitting (gap < 0.05)")
elif abs(gap_ens) < 0.10:
    print(f"    AUC Gap = {gap_ens:+.4f} → Mild variance (acceptable for surveillance)")
else:
    print(f"    AUC Gap = {gap_ens:+.4f} → Overfitting detected — consider regularisation")

print()


THESIS RESULTS SUMMARY — Final Model Evaluation

  Metric                                      RF    Bi-LSTM   Ensemble
  -----------------------------------------------------------------
  Train AUC                               0.9865     0.8954     0.9614
  Test AUC                                0.8990     0.9020     0.9121
  AUC Gap                                 0.0874    -0.0066     0.0493
  Test AP                                 0.8980     0.9080     0.9171

  Overfitting Verdict (Ensemble):
    AUC Gap = +0.0493 → NO overfitting (gap < 0.05)



---
## Section 6b – All-Models Summary Statistics Table (Thesis Export)

A single, publication-ready table comparing all 7 candidate models on the
held-out **test set** at the 0.45 decision threshold.  
The **Ensemble (RF + LSTM)** row is highlighted in gold as the selected final model.

Saved to `reports/figures/fig_all_models_summary_table.png`.

In [14]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd
import numpy as np
import os

# ── Build / refresh the full metrics table ──────────────────────────────────
# comp_df was built in Cell 10 (sorted by Test AUC desc, 1-indexed)
# Re-build here so this cell is fully self-contained when re-run.
_rows = []
for name, r in results.items():
    p_te = r['probs_te']
    y_te = r['y_te']
    p_tr = r['probs_tr']
    y_tr = r['y_tr']
    preds = (p_te >= THRESHOLD).astype(int)
    from sklearn.metrics import confusion_matrix, brier_score_loss
    cm = confusion_matrix(y_te, preds)
    tn, fp, fn, tp = cm.ravel()
    sens  = tp / (tp + fn)
    spec  = tn / (tn + fp)
    prec  = tp / (tp + fp) if (tp + fp) > 0 else 0
    f1_hr = 2 * prec * sens / (prec + sens) if (prec + sens) > 0 else 0
    brier = brier_score_loss(y_te, p_te)
    train_auc = roc_auc_score(y_tr, p_tr)
    test_auc  = roc_auc_score(y_te, p_te)
    test_ap   = average_precision_score(y_te, p_te)
    _rows.append(dict(
        Model       = name,
        train_auc   = train_auc,
        test_auc    = test_auc,
        auc_gap     = train_auc - test_auc,
        test_ap     = test_ap,
        sensitivity = sens,
        specificity = spec,
        f1_hr       = f1_hr,
        brier       = brier,
    ))

tbl = (pd.DataFrame(_rows)
         .sort_values('test_auc', ascending=False)
         .reset_index(drop=True))
tbl.index = tbl.index + 1   # 1-indexed rank

# ── Pretty-print to notebook output ─────────────────────────────────────────
print('=' * 110)
print('ALL-MODELS SUMMARY TABLE — Test-set performance at threshold = 0.45')
print('=' * 110)
header = f'{"Rank":>4}  {"Model":<30}  {"Train AUC":>9}  {"Test AUC":>8}  {"AUC Gap":>7}  '\
         f'{"PR-AUC":>6}  {"Sens":>5}  {"Spec":>5}  {"F1-HR":>5}  {"Brier":>5}'
print(header)
print('-' * 110)
for rank, row in tbl.iterrows():
    star = ' **  SELECTED' if row['Model'] == 'Ensemble (RF + LSTM)' else ''
    print(f'{rank:>4}  {row["Model"]:<30}  '
          f'{row["train_auc"]:>9.4f}  {row["test_auc"]:>8.4f}  '
          f'{row["auc_gap"]:>+7.4f}  {row["test_ap"]:>6.4f}  '
          f'{row["sensitivity"]:>5.3f}  {row["specificity"]:>5.3f}  '
          f'{row["f1_hr"]:>5.3f}  {row["brier"]:>5.4f}{star}')
print('=' * 110)
print()

# ── Produce a thesis-ready figure ────────────────────────────────────────────
# Column labels
col_labels = [
    'Rank', 'Model',
    'Train\nAUC', 'Test\nAUC', 'AUC\nGap',
    'PR-AUC\n(Avg Prec)', 'Sensitivity\n(HR Recall)',
    'Specificity', 'F1\n(High-Risk)', 'Brier\nScore'
]

cell_data = []
for rank, row in tbl.iterrows():
    cell_data.append([
        str(rank),
        row['Model'],
        f"{row['train_auc']:.4f}",
        f"{row['test_auc']:.4f}",
        f"{row['auc_gap']:+.4f}",
        f"{row['test_ap']:.4f}",
        f"{row['sensitivity']:.3f}",
        f"{row['specificity']:.3f}",
        f"{row['f1_hr']:.3f}",
        f"{row['brier']:.4f}",
    ])

n_rows = len(cell_data)
n_cols = len(col_labels)

fig, ax = plt.subplots(figsize=(18, 0.55 * n_rows + 1.8))
ax.axis('off')

tbl_widget = ax.table(
    cellText  = cell_data,
    colLabels = col_labels,
    loc       = 'center',
    cellLoc   = 'center',
)
tbl_widget.auto_set_font_size(False)
tbl_widget.set_fontsize(9.5)

# --- colour palette ---
HEADER_BG = '#1f3a5f'         # dark navy
HEADER_FG = 'white'
BEST_BG   = '#ffd966'         # gold for the ensemble / best row
BEST_FG   = '#1f1f1f'
ODD_BG    = '#e8f0fe'
EVEN_BG   = '#ffffff'
BORDER    = '#9ab3d4'

best_model_name = tbl.iloc[0]['Model']   # already sorted, rank 1 = best

for (row_idx, col_idx), cell in tbl_widget.get_celld().items():
    cell.set_edgecolor(BORDER)
    cell.set_linewidth(0.6)

    if row_idx == 0:                    # header row
        cell.set_facecolor(HEADER_BG)
        cell.set_text_props(color=HEADER_FG, fontweight='bold', fontsize=9)
        cell.set_height(0.16)
    else:                               # data rows
        model_in_row = cell_data[row_idx - 1][1]
        if model_in_row == best_model_name:
            cell.set_facecolor(BEST_BG)
            cell.set_text_props(color=BEST_FG, fontweight='bold')
        elif row_idx % 2 == 0:
            cell.set_facecolor(ODD_BG)
        else:
            cell.set_facecolor(EVEN_BG)

    # widen the Model column
    if col_idx == 1:
        cell.set_width(0.20)
    elif col_idx == 0:
        cell.set_width(0.04)

# bold the best value in each numeric column
metric_better = {
    2: 'max',  # Train AUC
    3: 'max',  # Test AUC
    4: 'min',  # AUC Gap (closest to 0)
    5: 'max',  # PR-AUC
    6: 'max',  # Sensitivity
    7: 'max',  # Specificity
    8: 'max',  # F1
    9: 'min',  # Brier (lower = better)
}
for col_idx, direction in metric_better.items():
    vals = [float(cell_data[r][col_idx]) for r in range(n_rows)]
    best_val = max(vals) if direction == 'max' else min(abs(v) for v in vals)
    for r in range(n_rows):
        v = float(cell_data[r][col_idx])
        if (direction == 'max' and v == best_val) or \
           (direction == 'min' and abs(v) == best_val):
            cell_obj = tbl_widget[r + 1, col_idx]
            txt = cell_obj.get_text()
            txt.set_fontweight('bold')

fig.suptitle(
    'Comparative Model Evaluation — All 7 Candidate Models\n'
    'Test-set metrics at decision threshold = 0.45  '
    '(highlighted row = selected final model)',
    fontsize=12, fontweight='bold', y=0.98
)

plt.tight_layout(rect=[0, 0, 1, 0.96])

OUT_DIR = '../../reports/figures'
os.makedirs(OUT_DIR, exist_ok=True)
save_path = os.path.join(OUT_DIR, 'fig_all_models_summary_table.png')
fig.savefig(save_path, dpi=200, bbox_inches='tight', facecolor='white')
plt.close(fig)
print(f'Thesis table saved  →  {save_path}')


ALL-MODELS SUMMARY TABLE — Test-set performance at threshold = 0.45
Rank  Model                           Train AUC  Test AUC  AUC Gap  PR-AUC   Sens   Spec  F1-HR  Brier
--------------------------------------------------------------------------------------------------------------
   1  Ensemble (RF + LSTM)               0.9614    0.9121  +0.0493  0.9171  0.836  0.821  0.824  0.1398 **  SELECTED
   2  Bi-LSTM + Attention                0.8954    0.9020  -0.0066  0.9080  0.856  0.761  0.809  0.1534
   3  Random Forest                      0.9865    0.8990  +0.0874  0.8980  0.823  0.808  0.810  0.1320
   4  XGBoost                            0.8938    0.8948  -0.0010  0.8976  0.804  0.843  0.814  0.1298
   5  LightGBM                           0.8939    0.8945  -0.0006  0.8971  0.812  0.832  0.815  0.1299
   6  CatBoost                           0.8840    0.8919  -0.0080  0.8937  0.812  0.829  0.813  0.1311
   7  Logistic Regression                0.8739    0.8904  -0.0164  0.8916  0.788

---
## Section 6c – Statistical Model Comparison: Pairwise DeLong AUC Tests

The **DeLong (1988)** test compares two correlated ROC-AUC values using the
placement-value (structural component) method.  All 7 models are aligned to
the **same LSTM-aligned test subset** (`idx_te`) so the test is valid for
correlated, paired comparisons.

- **H₀**: AUC_A = AUC_B  
- **H₁**: AUC_A ≠ AUC_B (two-sided)
- Bonferroni correction applied for 21 simultaneous tests: α′ = 0.05 / 21 ≈ 0.0024

Figure saved to `reports/figures/fig_delong_test.png`.

In [15]:
import numpy as np
from scipy import stats
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd
import os

# ═══════════════════════════════════════════════════════════════════════════
# DeLong (1988) test — exact, paired AUC comparison
# Reference: DeLong, DeLong & Clarke-Pearson, Biometrics 44:837-845 (1988)
# ═══════════════════════════════════════════════════════════════════════════
def _placement_values(y_true, y_score):
    """Compute V10 (per-positive) and V01 (per-negative) placement values."""
    pos = y_score[y_true == 1]
    neg = y_score[y_true == 0]
    n_pos, n_neg = len(pos), len(neg)
    # vectorised psi() using broadcasting — O(n_pos * n_neg)
    psi = (pos[:, None] > neg[None, :]).astype(float) \
        + 0.5 * (pos[:, None] == neg[None, :]).astype(float)
    V10 = psi.mean(axis=1)   # shape (n_pos,)
    V01 = psi.mean(axis=0)   # shape (n_neg,)
    return V10, V01, n_pos, n_neg

def delong_pvalue(y_true, score_a, score_b):
    """
    Two-sided DeLong test for paired AUCs.
    Returns (p_value, auc_a, auc_b).
    """
    y_true  = np.asarray(y_true, dtype=int)
    score_a = np.asarray(score_a, dtype=float)
    score_b = np.asarray(score_b, dtype=float)
    V10_a, V01_a, n, m = _placement_values(y_true, score_a)
    V10_b, V01_b, _, _ = _placement_values(y_true, score_b)
    auc_a = float(V10_a.mean())
    auc_b = float(V10_b.mean())
    # Var(AUC_a - AUC_b) = Var(V10_a - V10_b)/n + Var(V01_a - V01_b)/m
    var_diff = (np.var(V10_a - V10_b, ddof=1) / n +
                np.var(V01_a - V01_b, ddof=1) / m)
    if var_diff <= 1e-12:
        return (1.0 if abs(auc_a - auc_b) < 1e-9 else 0.0), auc_a, auc_b
    z = (auc_a - auc_b) / np.sqrt(var_diff)
    p = 2.0 * (1.0 - stats.norm.cdf(abs(z)))
    return p, auc_a, auc_b

# ── Align ALL models to the LSTM-aligned test subset ───────────────────────
# idx_te contains original DataFrame index labels of the aligned test rows.
# ML models were trained on X_train; we extract aligned test rows via X_test.iloc[idx_te].
print('Aligning all models to LSTM-compatible test subset …')
y_aligned = np.asarray(y_te_lstm, dtype=int)

aligned_probs = {}
for key, model in ml_models.items():
    name = display_names.get(key, key)
    X_al = (X_test.iloc[idx_te][model.feature_names_in_]
            if hasattr(model, 'feature_names_in_')
            else X_test.iloc[idx_te])
    aligned_probs[name] = model.predict_proba(X_al)[:, 1]

aligned_probs['Bi-LSTM + Attention']  = np.asarray(results['Bi-LSTM + Attention']['probs_te'])
aligned_probs['Ensemble (RF + LSTM)'] = np.asarray(results['Ensemble (RF + LSTM)']['probs_te'])

# Order models by descending aligned Test AUC
names_ord = sorted(aligned_probs,
                   key=lambda k: roc_auc_score(y_aligned, aligned_probs[k]),
                   reverse=True)
auc_vals  = [roc_auc_score(y_aligned, aligned_probs[n]) for n in names_ord]
n_models  = len(names_ord)
N_TESTS   = n_models * (n_models - 1) // 2   # 21
ALPHA     = 0.05
ALPHA_B   = ALPHA / N_TESTS                   # Bonferroni correction

print(f'  Aligned subset size : {y_aligned.sum()} positive / {(1-y_aligned).sum()} negative')
print(f'  Models evaluated    : {n_models}')
print(f'  Pairwise tests      : {N_TESTS}')
print(f'  Bonferroni α\u2032        : {ALPHA_B:.5f}\n')

# ── Build p-value matrix ────────────────────────────────────────────────────
pmat = np.ones((n_models, n_models))
for i in range(n_models):
    for j in range(i + 1, n_models):
        p, _, _ = delong_pvalue(y_aligned,
                                aligned_probs[names_ord[i]],
                                aligned_probs[names_ord[j]])
        pmat[i, j] = pmat[j, i] = p

# ── Console output: full pair table ─────────────────────────────────────────
W = 106
print('=' * W)
print('DELONG PAIRWISE AUC COMPARISON  (all models on LSTM-aligned test subset)')
print('=' * W)
print(f'{"Model A":<28}  {"Model B":<28}  {"AUC-A":>6}  {"AUC-B":>6}  {"Δ AUC":>7}  {"p-value":>9}  Sig.')
print('-' * W)
for i in range(n_models):
    for j in range(i + 1, n_models):
        p_val = pmat[i, j]
        delta = auc_vals[i] - auc_vals[j]
        sig   = '***' if p_val < ALPHA_B else ('*  ' if p_val < ALPHA else 'n.s.')
        print(f'{names_ord[i]:<28}  {names_ord[j]:<28}  '
              f'{auc_vals[i]:>6.4f}  {auc_vals[j]:>6.4f}  '
              f'{delta:>+7.4f}  {p_val:>9.4f}  {sig}')
print('=' * W)
print(f'*** p < {ALPHA_B:.4f} (Bonferroni)   * p < {ALPHA} (uncorrected)   n.s. not significant')

# ── Verdict ─────────────────────────────────────────────────────────────────
best = names_ord[0]
sig_raw  = sum(pmat[0, j] < ALPHA   for j in range(1, n_models))
sig_bonf = sum(pmat[0, j] < ALPHA_B for j in range(1, n_models))
print()
print('STATISTICAL VERDICT')
print('-' * 60)
print(f'  Best model  : {best}')
print(f'  Aligned AUC : {auc_vals[0]:.4f}')
print(f'  Significantly better (p<{ALPHA})        : vs {sig_raw}/{n_models-1} competitors')
print(f'  Significantly better (Bonferroni p<{ALPHA_B:.4f}): vs {sig_bonf}/{n_models-1} competitors')

# ── Heatmap figure ───────────────────────────────────────────────────────────
short = {
    'Ensemble (RF + LSTM)' : 'Ensemble\n(RF+LSTM)',
    'Bi-LSTM + Attention'  : 'Bi-LSTM\n+Attn',
    'Random Forest'        : 'Random\nForest',
    'XGBoost'              : 'XGBoost',
    'LightGBM'             : 'LightGBM',
    'CatBoost'             : 'CatBoost',
    'Logistic Regression'  : 'Logistic\nReg.',
}
tick_lbls = [f"{short.get(n, n)}\n({auc_vals[i]:.4f})" for i, n in enumerate(names_ord)]

fig, ax = plt.subplots(figsize=(12, 9))

# colour: dark-green = significant (small p), pale = not significant
cmap       = plt.cm.RdYlGn_r
diag_mask  = np.eye(n_models, dtype=bool)
masked_p   = np.where(diag_mask, np.nan, pmat)
im = ax.imshow(masked_p, cmap=cmap, vmin=0.0, vmax=0.25, aspect='auto')

for i in range(n_models):
    for j in range(n_models):
        if i == j:
            ax.text(j, i, '—', ha='center', va='center', fontsize=11, color='#666')
        else:
            p_val = pmat[i, j]
            stars = ('***' if p_val < ALPHA_B else
                     ('*'  if p_val < ALPHA   else ''))
            label = f"{p_val:.3f}\n{stars}" if stars else f"{p_val:.3f}"
            fc    = 'white' if p_val < 0.05 else '#1a1a1a'
            ax.text(j, i, label, ha='center', va='center', fontsize=8.5,
                    color=fc, fontweight='bold' if stars else 'normal')

ax.set_xticks(range(n_models))
ax.set_yticks(range(n_models))
ax.set_xticklabels(tick_lbls, fontsize=8.5, ha='center')
ax.set_yticklabels(tick_lbls, fontsize=8.5, ha='right')
ax.tick_params(axis='x', pad=4)

# highlight the best model's row/column borders
for spine in ['top','bottom','left','right']:
    ax.spines[spine].set_linewidth(0)
for tick in ax.get_xticklabels():
    if '(' + f"{auc_vals[0]:.4f}" + ')' in tick.get_text():
        tick.set_color('#c85000')
        tick.set_fontweight('bold')
for tick in ax.get_yticklabels():
    if '(' + f"{auc_vals[0]:.4f}" + ')' in tick.get_text():
        tick.set_color('#c85000')
        tick.set_fontweight('bold')

cbar = plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label('p-value  (DeLong test)', fontsize=10)
cbar.ax.axhline(ALPHA,   color='blue',   linewidth=1.2, linestyle='--')
cbar.ax.axhline(ALPHA_B, color='purple', linewidth=1.2, linestyle=':')
cbar.ax.text(1.6, ALPHA,   f'  p={ALPHA}',   va='center', fontsize=8, color='blue')
cbar.ax.text(1.6, ALPHA_B, f'  p={ALPHA_B:.4f}\n  (Bonf)', va='center', fontsize=7, color='purple')

ax.set_title(
    'DeLong Pairwise AUC Comparison — All 7 Candidate Models\n'
    f'Evaluated on LSTM-aligned test subset  |  '
    f'*** p < {ALPHA_B:.4f} (Bonferroni, N=21 tests)   * p < 0.05  (row vs column)\n'
    f'Models ranked by descending AUC  |  Orange = best model',
    fontsize=11, fontweight='bold', pad=16)

plt.tight_layout()
OUT = '../../reports/figures'
os.makedirs(OUT, exist_ok=True)
out_path = os.path.join(OUT, 'fig_delong_test.png')
fig.savefig(out_path, dpi=160, bbox_inches='tight', facecolor='white')
plt.close(fig)
print(f'\nDeLong heatmap saved → {out_path}')


Aligning all models to LSTM-compatible test subset …
  Aligned subset size : 444 positive / 481 negative
  Models evaluated    : 7
  Pairwise tests      : 21
  Bonferroni α′        : 0.00238

DELONG PAIRWISE AUC COMPARISON  (all models on LSTM-aligned test subset)
Model A                       Model B                        AUC-A   AUC-B    Δ AUC    p-value  Sig.
----------------------------------------------------------------------------------------------------------
Ensemble (RF + LSTM)          Random Forest                 0.9121  0.9058  +0.0063     0.0444  *  
Ensemble (RF + LSTM)          Bi-LSTM + Attention           0.9121  0.9020  +0.0101     0.0048  *  
Ensemble (RF + LSTM)          XGBoost                       0.9121  0.8991  +0.0131     0.0017  ***
Ensemble (RF + LSTM)          LightGBM                      0.9121  0.8981  +0.0140     0.0009  ***
Ensemble (RF + LSTM)          CatBoost                      0.9121  0.8969  +0.0153     0.0004  ***
Ensemble (RF + LSTM)       

---
## Section 7 – Ensemble-Context SHAP Feature Importance (XAI)

SHAP values are computed via `shap.TreeExplainer` on the **Random Forest**
component of the ensemble model. Justification:
- RF contributes **40 %** of the final ensemble probability.
- `TreeExplainer` gives **exact** SHAP values (vs model-agnostic approximations).
- Empirically the RF and Bi-LSTM feature rankings are highly correlated,
  so RF SHAP serves as a robust proxy for the full ensemble.

Figures are saved to `../../reports/figures/MLModels/` and loaded automatically
by the Streamlit dashboard **Feature Insights** tab.

In [16]:
import shap
import numpy as np
import matplotlib
matplotlib.use('Agg')        # headless – no GUI popup
import matplotlib.pyplot as plt
import os

SHAP_OUT = '../../reports/figures/MLModels'
os.makedirs(SHAP_OUT, exist_ok=True)
TOP_N = 15

# ── 7.1  TreeExplainer on RF component ──────────────────────────────────────
print('Building TreeExplainer on RF component …')
rf_model_local = ml_models.get('rf', ml_models.get(list(ml_models.keys())[0]))
explainer = shap.TreeExplainer(rf_model_local)

# Use LSTM-aligned test rows for consistent sample indexing
if hasattr(rf_model_local, 'feature_names_in_'):
    X_rf_shap = X_test.iloc[idx_te][rf_model_local.feature_names_in_]
else:
    X_rf_shap = X_test.iloc[idx_te]

shap_raw = explainer.shap_values(X_rf_shap)

# Normalise to positive-class slice (handles list / 3-D / 2-D arrays)
if isinstance(shap_raw, list):
    shap_values = shap_raw[1]
elif np.array(shap_raw).ndim == 3:
    shap_values = np.array(shap_raw)[:, :, 1]
else:
    shap_values = np.array(shap_raw)

feature_names = list(X_rf_shap.columns)
mean_abs_shap = np.abs(shap_values).mean(axis=0)
top_idx       = np.argsort(mean_abs_shap)[::-1][:TOP_N]
top_features  = [feature_names[i] for i in top_idx]

print(f'Top-{TOP_N} features by mean |SHAP|:')
for rank, i in enumerate(top_idx, 1):
    print(f'  {rank:2d}. {feature_names[i]:45s}  {mean_abs_shap[i]:.4f}')

# ── 7.2  Bar chart  ──────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))
vals   = mean_abs_shap[top_idx[::-1]]
labels = [feature_names[i] for i in top_idx[::-1]]
ax.barh(labels, vals, color='#60A5FA', edgecolor='white', linewidth=0.5)
ax.set_xlabel('Mean |SHAP value|  (impact on ensemble risk probability)', fontsize=11)
ax.set_title(
    f'Ensemble-Context SHAP – Top-{TOP_N} Feature Importance\n'
    '(RF component, 40 % of final ensemble probability)', fontsize=12)
ax.tick_params(axis='y', labelsize=9)
plt.tight_layout()
path_bar = os.path.join(SHAP_OUT, 'fig6a_shap_bar.png')
fig.savefig(path_bar, dpi=150, bbox_inches='tight')
plt.close(fig)
print(f'Saved: {path_bar}')

# ── 7.3  Beeswarm plot ───────────────────────────────────────────────────────
shap_top = shap_values[:, top_idx]
X_top    = X_rf_shap.iloc[:, top_idx].copy()
X_top.columns = top_features

shap.summary_plot(
    shap_top, X_top,
    feature_names=top_features,
    plot_type='dot',
    max_display=TOP_N,
    show=False,
    color_bar_label='Feature value'
)
plt.title(
    f'Ensemble-Context SHAP Beeswarm – Direction & Magnitude\n'
    '(RF component, 40 % of final ensemble probability)', fontsize=12)
plt.tight_layout()
path_bee = os.path.join(SHAP_OUT, 'fig6b_shap_beeswarm.png')
plt.savefig(path_bee, dpi=150, bbox_inches='tight')
plt.close('all')
print(f'Saved: {path_bee}')

# ── 7.4  Dependence plots for top-3 features ────────────────────────────────
dep_features = [
    ('HR_lag1',            'fig6c_shap_dep_1_HR_lag1.png'),
    ('HR_roll3',           'fig6c_shap_dep_2_HR_roll3.png'),
    ('District_risk_rate', 'fig6c_shap_dep_3_District_risk_rate.png'),
]
for feat_name, fname in dep_features:
    if feat_name not in feature_names:
        print(f'  Skipping dependence plot for {feat_name} (not in RF features)')
        continue
    fig, ax = plt.subplots(figsize=(8, 5))
    feat_col_idx = feature_names.index(feat_name)
    shap.dependence_plot(
        feat_col_idx, shap_values, X_rf_shap,
        feature_names=feature_names,
        ax=ax, show=False
    )
    ax.set_title(
        f'SHAP Dependence: {feat_name}\n'
        '(Ensemble-context via RF component)', fontsize=11)
    plt.tight_layout()
    path_dep = os.path.join(SHAP_OUT, fname)
    fig.savefig(path_dep, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'Saved: {path_dep}')

print('\nAll SHAP / XAI figures saved to', SHAP_OUT)


Building TreeExplainer on RF component …


KeyboardInterrupt: 